# Construção de sistema de RAG utilizando Ollama

## 1) Carregamento de bibliotecas

In [13]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer
import os

In [2]:
os.chdir(r'c:\Users\francisco.bneto\Documents\gen-ai-formation')
print(os.getcwd())

c:\Users\francisco.bneto\Documents\gen-ai-formation


## 2) Carregamento de dados

In [3]:
pdfs = DirectoryLoader("./documentos", glob="*.pdf").load()

len(pdfs)

3

## 3) Criação dos chunks

In [5]:
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

c:\Users\francisco.bneto\AppData\Local\miniconda3\envs\genai-py314\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\francisco.bneto\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [7]:
splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=1250,
    chunk_overlap=150
)

chunks = splitter.split_documents(pdfs)

len(chunks)

37

## 4) Criação do banco vetorial

In [10]:
embed_model= OllamaEmbeddings(
    model="bge-m3:567m"
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embed_model
)

In [15]:
prompt = """
Você é um especialista em questões bancárias, especialmente em dúvidas relacionadas a cartões de crédito.

Responsa as perguntas usando exclusivamente os conteúdos fornecidos.

Contexto:
{contexto}
"""

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", prompt),
        ("human", "{query}")
    ]
)

In [17]:
retriever = vector_store.as_retriever()

llm_model = OllamaLLM(
    model="gemma3:4b"
)

chain = prompt_template | llm_model | StrOutputParser()

In [18]:
query = "como fazer um seguro viagem?"

llm_model.invoke(query)

'Fazer um seguro viagem pode parecer complicado no início, mas com as informações corretas, o processo se torna bem simples e tranquilo. Aqui está um guia completo de como fazer um seguro viagem, abordando os principais passos e aspectos a serem considerados:\n\n**1. Entenda a Necessidade:**\n\n*   **Por que você precisa de um seguro viagem?** Considere o destino, duração da viagem, atividades que pretende realizar e o seu perfil de risco. Viajar sozinho, para um destino com sistema de saúde diferente do seu ou para atividades de risco (esportes radicais, etc.) geralmente exigem um seguro mais completo.\n*   **O que o seguro cobre?** Verifique quais coberturas são importantes para você:\n    *   **Assistência médica e hospitalar:** Cobertura em caso de doença, acidente, emergências médicas, tratamento odontológico, etc.\n    *   **Assistência odontológica:** Cobertura para emergências odontológicas.\n    *   **Traslado:** Deslocamento para hospitais ou clínicas.\n    *   **Repatriação: